In [ ]:

import pandas as pd
from pathlib import Path
import calendar

data_dir = Path("../data/raw")

files = {
    8: "data_barrages_tensift_aout_2026.xlsx",
    9: "data_barrages_tensift_sept_2026.xlsx",
    10: "data_barrages_tensift_oct_2026.xlsx"
}

def clean_dams(file_path, month) -> pd.DataFrame:

    df = pd.read_excel(file_path, header=None)
    file_dams_data = []

    # Find the row containing "JOUR"
    header_rows = df.index[
        df.iloc[:, 0].astype(str).str.strip() == "JOUR"
    ]
    if len(header_rows) != 1 or header_rows[0] < 1:
        raise ValueError(f"{file_path}: expected one JOUR header after a capacity row")
    if df.shape[1] < 11:
        raise ValueError(f"{file_path}: expected five reserve/fill-rate column pairs")
    header_row = header_rows[0]

    capacity_row = header_row - 1
    start_row = header_row + 1

    # Extract days and select valid day rows
    jours = pd.to_numeric(
        df.iloc[start_row:, 0], errors="coerce"
    )

    max_day = calendar.monthrange(2026, month)[1]
    valid = jours.between(1, max_day) & jours.mod(1).eq(0)

    for i in range(1, 11, 2):

        dam_name = str(df.iloc[0, i]).strip()

        capacite = pd.to_numeric(
            df.iloc[capacity_row, i], errors="coerce"
        )

        reserve = pd.to_numeric(
            df.iloc[start_row:, i], errors="coerce"
        )

        taux = pd.to_numeric(
            df.iloc[start_row:, i + 1], errors="coerce"
        )

        clean_data = pd.DataFrame({
            "jour": jours[valid].reset_index(drop=True).astype(int),
            "dam": dam_name,
            "capacity_mm3": capacite,
            "reserve_mm3": reserve[valid].reset_index(drop=True),
            "fill_rate": taux[valid].reset_index(drop=True)
        })

        clean_data["date"] = pd.to_datetime({
            "year": 2026,
            "month": month,
            "day": clean_data["jour"]
        })

        clean_data = clean_data[
            ["date", "dam", "capacity_mm3",
             "reserve_mm3", "fill_rate"]
        ]

        file_dams_data.append(clean_data)

    return pd.concat(file_dams_data, ignore_index=True)







In [23]:
monthly_data = []

for month, filename in files.items():
    file_path = data_dir / filename

    monthly_df = clean_dams(file_path, month)
    monthly_data.append(monthly_df)

all_dams = pd.concat(monthly_data, ignore_index=True)


all_dams = all_dams.sort_values(
    ["dam", "date"]
).reset_index(drop=True)



In [ ]:
print("Shape:", all_dams.shape)
print("\nFirst rows:")
print(all_dams.head())
print("\nMissing values:")
print(all_dams.isna().sum())
print("\nUnique dams:", all_dams["dam"].nunique())
print("\nDuplicates:", all_dams.duplicated(
    subset=["date", "dam"]
).sum())


if all_dams.duplicated(subset=["date", "dam"]).any():
    raise ValueError("Duplicate dam/date observations; processed CSV was not written")

output = Path("../data/processed/tensift_cleaned.csv")
output.parent.mkdir(parents=True, exist_ok=True)

all_dams.to_csv(output, index=False)

In [ ]:
for month, monthly_df in zip(files, monthly_data):
    print(f"Month {month}:")
    print(monthly_df)